# Dự án 5 — Máy tự khám phá — Đề D: Mê cung Q-learning — thiết kế phần thưởng
## BẢN CỦA NHÓM

**Trường THCS và THPT Đinh Thiện Lý — Machine Learning Level 1 — Khối 10**

---

Không có bảng dữ liệu, không có đáp án: robot **tự thử và sai** trong mê cung 6 × 6 (như Bài 28, thêm đồng xu). Nhóm là người **thiết kế phần thưởng**: thưởng / phạt thế nào thì robot học đúng việc mình muốn?

**Câu hỏi lớn của dự án:** *Không có nhãn, máy tìm ra điều gì? Không có đáp án, robot học thế nào?*

**Dữ liệu:** Mê cung tự tạo trong notebook (không cần tải dữ liệu)  

### Các mốc (nộp trên Canvas trước 23:59 Thứ 7 của tuần đó)

| Tuần | Mốc | Nộp |
|---|---|---|
| 1 · buổi 22 | **M1 + M2** Chọn cột, chọn k (hoặc cho robot học), đặt tên nhóm | Notebook bước 1 – 4 |
| 2 · buổi 23 | **M3 + M4** Kiểm tra / thí nghiệm, 1 slide | Notebook đủ + slide + Nhật ký AI |

### Nhóm và vai — **đổi vai mỗi tuần**

Nhóm: … · Lớp: …

| Vai | Tuần 1 | Tuần 2 | Tuần 3 | Việc chính |
|---|---|---|---|---|
| Lái code | … | … | … | Gõ và chạy notebook; không ai gõ hộ hai tuần liền |
| Nghiên cứu | … | … | … | Đọc bài thư viện, hỏi Gemini “vì sao”, ghi Nhật ký AI |
| Thư ký số liệu | … | … | … | Ghi bảng thí nghiệm, kiểm tra con số trước khi báo cáo |
| Trình bày | … | … | … | Báo cáo 1 phút mỗi tuần, làm slide / poster |

### Quy tắc dùng AI

- **Được:** hỏi Gemini (✨ trong Colab) một lệnh làm gì, bấm *Giải thích lỗi*, hỏi *vì sao* làm bước này, nhờ gợi ý cách tiếp cận.
- **Không được:** nộp con số, biểu đồ, kết luận mà nhóm không tự chạy ra; để AI viết cả notebook.
- Ô code do AI gợi ý ghi dòng đầu `# AI gợi ý — nhóm đã kiểm tra: ...`. Mỗi tuần ghi **Nhật ký AI** ở cuối.
- Khi đi qua nhóm và ở mốc cuối, thầy gọi **một bạn bất kỳ** giải thích một dòng code hoặc một con số.

### Dữ liệu

| Cột | Ý nghĩa |
|---|---|
| `S, G` | Xuất phát, đích (+10) |
| `H` | Hố — kết thúc tập, phạt `phat_ho` |
| `$` | Đồng xu, thưởng `thuong_xu`, nhặt lại được |
| `#` | Tường |
| `mỗi bước` | −1 điểm |


---
# TUẦN 1 (buổi 22) — Robot học trong mê cung

**Mốc của tuần:** **M1 + M2** Chọn cột, chọn k (hoặc cho robot học), đặt tên nhóm  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 28 Học tăng cường](https://vuongndlst.github.io/ml-level1/bai28/)


---
## Bước 0 — Nạp dữ liệu (đã viết sẵn, chỉ cần chạy)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# S = xuất phát, G = đích (+10), H = hố, $ = đồng xu, # = tường; mỗi bước −1 (như Bài 28, thêm ô đồng xu)
LUOI = ["S.....",
        ".####.",
        "..H...",
        "#.#.#.",
        "...H..",
        ".#.$.G"]
HUONG = ["lên", "xuống", "trái", "phải"]
DI = [(-1, 0), (1, 0), (0, -1), (0, 1)]
N = len(LUOI)
def tim(ky):
    return [(r, c) for r in range(N) for c in range(N) if LUOI[r][c] == ky][0]
BAT_DAU, DICH = tim("S"), tim("G")
def buoc(o, a, phat_ho=-10, thuong_xu=0):
    """Đi một bước từ ô o theo hướng a -> (ô mới, phần thưởng, kết thúc?). Đồng xu nhặt lại được mãi."""
    r, c = o[0] + DI[a][0], o[1] + DI[a][1]
    if not (0 <= r < N and 0 <= c < N) or LUOI[r][c] == "#":
        return o, -1, False
    if LUOI[r][c] == "H":
        return (r, c), phat_ho, True
    if LUOI[r][c] == "G":
        return (r, c), 10, True
    if LUOI[r][c] == "$":
        return (r, c), thuong_xu, False
    return (r, c), -1, False
def hoc(so_tap=300, eps=0.1, alpha=0.5, gamma=0.9, seed=0, toi_da=100, **pt):
    """Q-learning (như Bài 28). pt = phat_ho, thuong_xu. Trả về bảng Q, số bước và tổng điểm mỗi tập."""
    rng = np.random.RandomState(seed)
    Q = np.zeros((N, N, 4))
    so_buoc, tong_diem = [], []
    for _ in range(so_tap):
        o, diem = BAT_DAU, 0
        for t in range(1, toi_da + 1):
            if rng.rand() < eps:
                a = rng.randint(4)                            # khám phá
            else:
                a = rng.choice(np.flatnonzero(Q[o] == Q[o].max()))   # khai thác
            o2, r, xong = buoc(o, a, **pt)
            Q[o][a] += alpha * ((r if xong else r + gamma * Q[o2].max()) - Q[o][a])
            o, diem = o2, diem + r
            if xong:
                break
        so_buoc.append(t)
        tong_diem.append(diem)
    return Q, so_buoc, tong_diem
def duong_di(Q, toi_da=30, **pt):
    """Đi theo bảng Q (luôn chọn điểm cao nhất) từ S -> (danh sách ô, kết quả)."""
    o, ds = BAT_DAU, [BAT_DAU]
    for _ in range(toi_da):
        o, r, xong = buoc(o, int(Q[o].argmax()), **pt)
        ds.append(o)
        if xong:
            return ds, ("tới đích" if o == DICH else "rơi hố")
    return ds, "lạc đường"
print("\n".join(LUOI))


### Câu hỏi của nhóm (M1)

*Gợi ý câu hỏi:* Cần bao nhiêu tập thì robot tới đích? · Phạt hố nhẹ thì sao? · Đồng xu có làm robot quên đích?

| | Nhóm điền |
|---|---|
| Câu hỏi nhóm chọn | … |
| Cột sẽ dùng | … |
| Dự đoán trước (giả thuyết) | … |
| Làm sao biết dự đoán đúng hay sai? | … |


---
## Bước 1 — Robot chưa học

> **🎯 Mục tiêu:** Cho robot đi ngẫu nhiên 100 lần: tới đích bao nhiêu lần?
>
> **🧭 Vì sao bước này?** Mốc của học tăng cường: robot chưa học làm được gì.
>
> **📋 Nhiệm vụ:**
> 1. Đọc mê cung, tìm đường ngắn nhất bằng mắt.
> 2. Chạy robot ngẫu nhiên 100 lần.
> 3. Đếm kết quả.
>
> **💡 Gợi ý lệnh:**
> - `rng.randint(4)` — chọn hướng ngẫu nhiên
>
> **✅ Đạt khi:** có bảng đếm tới đích / rơi hố / lạc đường.


In [ ]:
# Buoc 1 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 2 — Cho robot học

> **🎯 Mục tiêu:** Q-learning 10, 30, 100, 300 tập; vẽ số bước theo tập.
>
> **🧭 Vì sao bước này?** Robot không có đáp án, chỉ có phần thưởng; học dần qua nhiều tập.
>
> **📋 Nhiệm vụ:**
> 1. Chạy `hoc(n)` với 4 giá trị n.
> 2. Mỗi lần: `duong_di(Q)` tới đích chưa?
> 3. Vẽ số bước theo tập.
>
> **💡 Gợi ý lệnh:**
> - `hoc(300)` — học 300 tập
> - `duong_di(Q)` — đi theo bảng Q
>
> **✅ Đạt khi:** có bảng số tập → kết quả, và đồ thị.


In [ ]:
# Buoc 2 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 3 — Đọc đường đi và bảng Q

> **🎯 Mục tiêu:** Vẽ đường robot chọn; đọc điểm Q ở ô xuất phát.
>
> **🧭 Vì sao bước này?** Bảng Q là “trí nhớ” của robot: ô nào, hướng nào đáng giá bao nhiêu.
>
> **📋 Nhiệm vụ:**
> 1. Vẽ đường đi bằng dấu *.
> 2. Đường có ngắn nhất không, có tránh hố không?
>
> **💡 Gợi ý lệnh:**
> - `Q[BAT_DAU]` — điểm 4 hướng ở ô xuất phát
>
> **✅ Đạt khi:** có mê cung với đường đi và một câu nhận xét.


In [ ]:
# Buoc 3 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 4 — Bảng thí nghiệm phần thưởng

> **🎯 Mục tiêu:** Đổi phạt hố, thưởng đồng xu, tỉ lệ khám phá — mỗi lần một thứ.
>
> **🧭 Vì sao bước này?** Trong học tăng cường, con người không viết đáp án mà viết **phần thưởng** — đó là quyết định lớn nhất.
>
> **📋 Nhiệm vụ:**
> 1. Phạt hố −10, −3, −1, 0.
> 2. Đồng xu 0, +1, +3.
> 3. Khám phá eps 0; 0,1; 0,5.
> 4. Ghi bảng kết quả.
>
> **💡 Gợi ý lệnh:**
> - `hoc(300, phat_ho=-1)` — đổi phần thưởng
> - `duong_di(Q, phat_ho=-1)` — đi với cùng phần thưởng
>
> **✅ Đạt khi:** có bảng ≥ 6 thí nghiệm.


In [ ]:
# Buoc 4 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M1 + M2 (tuần 1)

- [ ] Bảng câu hỏi của nhóm
- [ ] Bước 1 – 3 có nhận xét
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 2 (buổi 23) — Robot học sai khi nào?

**Mốc của tuần:** **M3 + M4** Kiểm tra / thí nghiệm, 1 slide  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 26 K-Means](https://vuongndlst.github.io/ml-level1/bai26/), [Bài 28 Học tăng cường](https://vuongndlst.github.io/ml-level1/bai28/)


---
## Bước 5 — Khi phần thưởng sai

> **🎯 Mục tiêu:** Xem kỹ robot với đồng xu +3; viết kết luận về thiết kế phần thưởng.
>
> **🧭 Vì sao bước này?** Robot tối đa điểm thưởng — không hiểu ý định của người thiết kế.
>
> **📋 Nhiệm vụ:**
> 1. In 8 ô cuối của đường đi.
> 2. Vì sao robot không về đích?
> 3. Sửa phần thưởng thế nào cho đúng?
>
> **💡 Gợi ý lệnh:**
> - `duong_di(Q, thuong_xu=3)` — đường đi với đồng xu +3
>
> **✅ Đạt khi:** có giải thích và một cách sửa.


In [ ]:
# Buoc 5 - code cua nhom



**Nhận xét của nhóm:** …


### Slide kết quả (M4) — một slide, 4 ô

| Ô | Nội dung |
|---|---|
| Câu hỏi | … |
| Một hình | … |
| Một con số | … |
| Giới hạn | … |

*Làm trên Canva hoặc Google Slides; dán link vào ô nộp Canvas cùng link notebook.*


---
## Bước 6 — Nhật ký AI

Không dùng AI thì ghi *Không dùng*. Kiểm chứng = nhóm tự chạy lại, đối chiếu số, sửa chỗ AI sai.

| Nhóm hỏi AI | AI trả lời / viết | Nhóm kiểm chứng và sửa |
|---|---|---|
| … | … | … |
| … | … | … |
| … | … | … |


---
## Mở rộng — nhóm xong sớm (không bắt buộc)

1. Đổi `gamma` (0,5 / 0,9 / 0,99): robot có “nhìn xa” hơn không?
2. Vẽ mê cung của nhóm (8 × 8, có hai đường tới đích) — robot chọn đường nào, vì sao?


In [ ]:
# Mo rong - code cua nhom



---
## Trước khi nộp

- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Mỗi bước có một ô nhận xét **có số và cỡ nhóm**
- [ ] Không có câu “… **gây ra** …” khi dữ liệu chỉ cho thấy đi cùng nhau
- [ ] Bảng vai trò và nhật ký AI đã điền
- [ ] Link notebook ở chế độ *Người xem*, dán lên Canvas cùng link Canva
